In [3]:
!pip install -q "langchain-docling[local]"

In [4]:
from langchain_docling import DoclingLoader

In [8]:
FILE_PATH = "/kaggle/input/datasets/priyankaadheli1/budget-speech/budget_speech.pdf"

loader = DoclingLoader(file_path=FILE_PATH)

documents = loader.load()

print(documents[0].page_content)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


2026-11-05 07:49:30,379 MatchingPostProcessor WARNING  1 of 295 pdf cells matched neither a row nor a column band of the 8x3 grid and were dropped from the table


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (566 > 512). Running this sequence through the model will result in indexing errors


SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026


# Text Splitter 

In [9]:
!pip install -q langchain-text-splitters

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = splitter.split_documents(documents)

print("Total chunks:", len(texts))

Total chunks: 213


In [12]:
from langchain_core.documents import Document

# Clean Metadata

In [13]:
def clean_metadata(doc):

    simple = {
        k: v
        for k, v in doc.metadata.items()
        if isinstance(v, (str, int, float, bool))
    }

    return Document(
        page_content=doc.page_content,
        metadata=simple
    )


clean_texts = [clean_metadata(d) for d in texts]

print(clean_texts[0].metadata)

{'source': '/kaggle/input/datasets/priyankaadheli1/budget-speech/budget_speech.pdf'}


In [14]:
!pip install -q langchain-huggingface sentence-transformers

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

In [16]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded


In [22]:
!pip install -q --upgrade langchain-chroma chromadb

In [25]:
!pip install -q --upgrade --force-reinstall \
    opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1 \
    opentelemetry-exporter-otlp-proto-grpc==1.42.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-exporter-otlp-common 0.66b2 requires opentelemetry-sdk~=1.45.2, but you have opentelemetry-sdk 1.42.1 which is incompatible.
grpcio-tools 1.85.1 requires protobuf<8.0.0,>=7.35.1, but you have protobuf 6.33.6 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [26]:
import chromadb
from langchain_chroma import Chroma

In [27]:
vector_store = Chroma.from_documents(
    documents=clean_texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Chroma DB created and persisted")

Chroma DB created and persisted


In [28]:
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

print("Chroma DB reloaded")

Chroma DB reloaded


# Similarity Search Test

In [29]:
query = "Biopharma SHAKTI outlay"

results = vector_store.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:

    print("Page:", doc.metadata.get("page"))

    print("Score:", score)

    print("Content:", doc.page_content)

    print("-" * 50)

Page: None
Score: 0.5278290510177612
Content: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet global standards and approval timeframes through a dedicated scientific rev

In [30]:
!pip install -q langchain-groq

In [33]:
from langchain_groq import ChatGroq

In [34]:
from kaggle_secrets import UserSecretsClient

In [39]:
user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

print("Groq API key loaded:", GROQ_API_KEY is not None)

Groq API key loaded: True


In [41]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

print("ChatGroq model loaded")

ChatGroq model loaded


In [42]:
from langchain_core.tools import tool

In [43]:
@tool
def document_search(query: str) -> str:
    """Search the Union Budget 2026-27 PDF for relevant information."""

    results = vector_store.similarity_search(
        query,
        k=3
    )

    if not results:
        return "No relevant information found in the document."

    output = []

    for doc in results:

        page = doc.metadata.get("page", "Unknown")

        output.append(
            f"Page {page}: {doc.page_content}"
        )

    return "\n\n".join(output)

# Test the search tool

In [44]:
result = document_search.invoke(
    {"query": "Biopharma SHAKTI outlay"}
)

print(result)

Page Unknown: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for domestic production of biologics and biosimilars. The Strategy will include a Biopharma-focused network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. It will also create a network of over 1000 accredited India Clinical Trials sites. We propose to strengthen the Central Drugs Standard Control Organisation to meet global standards and approval timeframes through a dedicated scientific review cadre and specialists. 17. I

In [45]:
!pip install -q langchain-core

In [46]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

In [47]:
prompt = ChatPromptTemplate.from_messages([

    (
        "system",
        """You are an Agentic RAG assistant for the Union Budget 2026-27.

        Instructions:

        1. Use the document_search tool for questions about the budget.
        2. Do not search for greetings or general knowledge questions.
        3. Answer using the retrieved document information.
        4. Cite the page number for every document-based answer.
        5. If the answer is not available in the document, say so.
        6. Never guess or invent information.
        7. Answer in simple and clear language."""
    ),

    MessagesPlaceholder(variable_name="history"),

    ("human", "{input}")

])

In [48]:
import ast
import operator

In [49]:
operations = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}


def safe_calculate(expression):

    def evaluate(node):

        if isinstance(node, ast.Constant):

            if type(node.value) in (int, float):
                return node.value

        if isinstance(node, ast.BinOp):

            op = type(node.op)

            if op in operations:

                return operations[op](
                    evaluate(node.left),
                    evaluate(node.right)
                )

        if isinstance(node, ast.UnaryOp):

            op = type(node.op)

            if op in operations:

                return operations[op](
                    evaluate(node.operand)
                )

        raise ValueError("Invalid expression")

    tree = ast.parse(expression, mode="eval")

    return evaluate(tree.body)

In [50]:
@tool
def calculator(expression: str) -> str:
    """Perform safe arithmetic calculations."""

    try:

        result = safe_calculate(expression)

        return str(result)

    except Exception:

        return "Invalid mathematical expression"

In [51]:
tools = [
    document_search,
    calculator
]

tool_map = {
    t.name: t
    for t in tools
}

llm_with_tools = llm.bind_tools(tools)

print("Both tools connected")

Both tools connected


In [52]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage
)

In [54]:
def agent_response(user_input, history, max_rounds=5):

    messages = prompt.invoke({
        "input": user_input,
        "history": history
    }).to_messages()

    searched = False
    used_tools = []

    for _ in range(max_rounds):

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        if not response.tool_calls:

            return response.content, searched, used_tools

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]

            tool_args = tool_call["args"]

            if tool_name == "document_search":
                searched = True

            if tool_name not in used_tools:
                used_tools.append(tool_name)

            tool_result = tool_map[tool_name].invoke(
                tool_args
            )

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return (
        "Maximum tool rounds reached.",
        searched,
        used_tools
    )

In [55]:
history = []

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":

        print("Chat ended.")
        break

    answer, searched, used_tools = agent_response(
        user_input,
        history
    )

    print("\nAssistant:", answer)

    print("Searched:", "Y" if searched else "N")

    print("Tools used:", used_tools)

    history.append(
        HumanMessage(content=user_input)
    )

    history.append(
        AIMessage(content=answer)
    )


You:  Hello



Assistant: Hello! How can I help you with the Union Budget 2026‑27?
Searched: N
Tools used: []



You:  What is sum of 2 and 5



Assistant: The sum of 2 and 5 is **7**.
Searched: N
Tools used: ['calculator']



You:  What are the new income tax slab rates in this budget?



Assistant: Maximum tool rounds reached.
Searched: Y
Tools used: ['document_search']



You:  exit


Chat ended.


In [56]:
!pip install -q pandas

In [57]:
import pandas as pd

In [58]:
test_queries = [
    "What is the outlay of Biopharma SHAKTI?",
    "How does the 2026-27 fiscal deficit compare with 2025-26?",
    "What changes for NCCD on chewing tobacco, and what is the effective duty?",
    "Hi, how are you?",
    "Explain fiscal deficit in simple words.",
    "What are the new income tax slab rates in this budget?",
    "By what percentage does public capex rise from BE 2025-26 to FY2026-27?",
    "What is the combined outlay of Biopharma SHAKTI, the Container Manufacturing scheme and CCUS?"
]

In [59]:
expected_search = [
    "Y",  # Biopharma SHAKTI outlay
    "Y",  # Fiscal deficit comparison
    "Y",  # NCCD and duty
    "N",  # Greeting
    "N",  # General concept
    "Y",  # Budget income tax slabs
    "Y",  # Public capex values
    "Y"   # Combined budget outlay
]

In [60]:
test_results = []

for query in test_queries:
    answer, searched, used_tools = agent_response(query, [])

    test_results.append({
        "Query": query,
        "Searched": "Y" if searched else "N",
        "Tools Used": ", ".join(used_tools) if used_tools else "None",
        "Final Answer": answer
    })

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4ak4azaet98ndqhq9a9htkj` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6624, Requested 1532. Please try again in 1.17s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [61]:
test_results

[{'Query': 'What is the outlay of Biopharma SHAKTI?',
  'Searched': 'Y',
  'Tools Used': 'document_search',
  'Final Answer': 'The Biopharma\u202fSHAKTI programme is proposed with an outlay of **₹\u202f10,000\u202fcrores** over the next five years【Page\u202f16】.'},
 {'Query': 'How does the 2026-27 fiscal deficit compare with 2025-26?',
  'Searched': 'Y',
  'Tools Used': 'document_search',
  'Final Answer': 'The Union Budget 2026‑27 projects the fiscal deficit to fall to **4.3\u202f% of GDP**.  \nIn the preceding year, the Revised Estimates for 2025‑26 had the fiscal deficit at **4.4\u202f% of GDP**.\n\n**Comparison:** The fiscal deficit is expected to be **0.1\u202fpercentage point lower** in 2026‑27 than it was in 2025‑26, indicating a modest improvement in fiscal balance.  \n\n*Source: Union Budget 2026‑27 (Fiscal deficit estimate for BE\u202f2026‑27\u202f=\u202f4.3\u202f% of GDP; RE\u202f2025‑26\u202f=\u202f4.4\u202f% of GDP).*'},
 {'Query': 'What changes for NCCD on chewing tobacco